In [0]:
from pyspark.sql import functions as F

# Use silver schema
spark.sql("USE plworkforce_catalog.002_silver")

# Read from Bronze
df = spark.table("plworkforce_catalog.001_bronze.accounts")

# Transform
df_clean = (df

    # Clean text columns
    .withColumn("account_code", F.upper(F.trim("account_code")))
    .withColumn("account_name", F.initcap(F.trim("account_name")))
    .withColumn("account_type", F.initcap(F.trim("account_type")))
    .withColumn("category", F.initcap(F.trim("category")))

    # Convert boolean (if needed)
    .withColumn("is_active", F.col("is_active").cast("boolean"))

    # Data quality
    .dropDuplicates(["account_id"])
    .filter(F.col("account_id").isNotNull())
)

# Write to Silver
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("accounts")
)

print("Silver accounts created")

#to display the data in silver accounts table 
display(spark.read.table("plworkforce_catalog.002_silver.accounts"))